In [2]:
from collections import namedtuple

import geopandas as gpd
import h3
import pandas as pd
import yaml
from shapely import from_wkt


In [3]:
with open("../config.budapest.yaml", "r") as fp:
    CRS = yaml.safe_load(fp)["crs"]

In [4]:
opts = namedtuple("options", ["output", "area"])

In [5]:
opts.output = "../data/osm/"
opts.area = "budapest"

In [15]:
amenities = pd.read_csv(
    f"{opts.output}/{opts.area}/amenities/amenities_cleaned.wkt.csv",
    engine="pyarrow",
)
amenities["geometry"] = amenities["geometry"].apply(from_wkt)
amenities = gpd.GeoDataFrame(amenities, geometry="geometry", crs=4326)

In [16]:
amenities["centroid"] = amenities["geometry"].to_crs(CRS).centroid
amenities["centroid"] = amenities["centroid"].to_crs(4326)
amenities["lon"] = amenities["centroid"].x
amenities["lat"] = amenities["centroid"].y

In [18]:
for res in range(8, 11):
    amenities[f"h3_{res}"] = amenities.apply(
        lambda x: h3.latlng_to_cell(x["lat"], x["lon"], res=res), axis=1
    )

In [19]:
amenities.drop(columns=["centroid", "lon", "lat", "geometry_type", "osm_type"]).to_csv(
    "../data/osm/budapest/amenities/amenities_with_h3.wkt.csv", index=False
)